# 语义内核

在这个代码示例中，您将使用 [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI框架来创建一个基础代理。

本示例的目标是向您展示我们在后续代码示例中实现不同代理模式时将使用的步骤。


## 导入所需的 Python 包


In [6]:
import os 
from typing import Annotated # 导入 Annotated，用于给函数的返回值添加描述，这对于AI模型理解工具的作用至关重要。
from openai import AsyncOpenAI # 导入异步 OpenAI 客户端，用于与模型进行非阻塞通信。

from dotenv import load_dotenv # 导入 load_dotenv，用于从 .env 文件中加载环境变量。


# -----------------------------------------------------
# Semantic Kernel (SK) 代理和连接器组件
# 说明:semantic_kernel (通常缩写为 SK) 是一个由微软开发的开源 SDK（软件开发工具包），
# 其核心作用是帮助开发者轻松地将大型语言模型 (LLM) 的能力集成到现有的应用和编程语言中。
# 它的目标是作为 LLM 的编排层 (Orchestration Layer)，让开发者能够构建出具有复杂逻辑
# 和工具调用能力的 AI 代理 (Agent) 和 智能应用。
# 擅长构建一个强大的、拥有多项工具和记忆的单一核心代理，由它来决定如何执行任务。强调工具集成
# -----------------------------------------------------
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread # 导入代理核心组件
# ChatCompletionAgent: SK中用于定义基于聊天的AI代理的类。
# ChatHistoryAgentThread: 用于管理代理的独立对话线程/历史记录。
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion 
# 导入 OpenAI 聊天完成连接器。
from semantic_kernel.functions import kernel_function 
# 导入 kernel_function 装饰器，用于将普通的Python方法声明为AI可以使用的工具。


## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场中提供的其他模型，以查看不同的结果。

为了使用 `Azure Inference SDK`（用于 GitHub Models 的 `base_url`），我们将在 Semantic Kernel 中使用 `OpenAIChatCompletion` 连接器。此外，还有其他 [可用连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion)，可以将 Semantic Kernel 用于其他模型提供商。


In [7]:
import random   

# -----------------------------------------------------
# 1. 定义一个简单的代理工具 (Plugin)
# -----------------------------------------------------

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""
    """一个包含随机度假目的地的插件/工具。"""

    def __init__(self):
        # List of vacation destinations
        # 存储所有可能的度假目的地列表
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        # 用于跟踪上一个目的地，以避免重复推荐
        self.last_destination = None

    # 使用 @kernel_function 装饰器，将这个方法暴露给 AI 模型作为工具
    # 使用 Annotated 为返回值添加详细的类型和描述，模型会利用这些信息来决定何时调用此函数
    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        # 复制目的地列表
        available_destinations = self.destinations.copy()
        # 如果有上一个目的地，且列表不止一个，则从可用列表中移除上一个目的地，确保新的目的地不同
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        # 从过滤后的列表中随机选择一个目的地
        destination = random.choice(available_destinations)

        # Update the last destination
        # 更新上一个目的地
        self.last_destination = destination

        # 返回随机选择的目的地字符串
        return destination

In [8]:
# -----------------------------------------------------
# 2. 初始化配置和连接器
# -----------------------------------------------------
load_dotenv() # 从当前目录加载 .env 文件中的环境变量

# 使用通义大模型
model_name="qwen-max"
client = AsyncOpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY"), 
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 使用GPT大模型
# model_name = "gpt-4o-mini"
# 创建 AsyncOpenAI 客户端实例
# client = AsyncOpenAI(
#     api_key=os.environ["GITHUB_TOKEN"],
#     base_url="https://models.inference.ai.azure.com/"
# )

# Create an AI Service that will be used by the `ChatCompletionAgent`
# 创建 AI 服务对象，作为 Semantic Kernel 与 Qwen-Max 模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    ai_model_id=model_name,
    async_client=client,
)


## 创建代理

下面我们创建名为 `TravelAgent` 的代理。

在这个示例中，我们使用了非常简单的指令。您可以更改这些指令，观察代理如何做出不同的响应。


In [9]:
# -----------------------------------------------------
# 3. 创建和配置 AI 代理
# ChatCompletionAgent 是 Semantic Kernel (SK) 框架中用于构建**核心 AI 代理（Agent）**的基础类。
# 它的主要职责是定义 AI 的身份、能力和行为，并将这些元素集成起来，以便能够响应用户的请求。
# -----------------------------------------------------
agent = ChatCompletionAgent(
    service=chat_completion_service, # 将聊天服务连接到代理
    plugins=[DestinationsPlugin()], # 将上面定义的工具/插件添加到代理中
    name="TravelAgent", # 为代理命名
    # 为代理设置系统指令 (Instructions)，定义它的角色和能力
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## 运行代理

现在我们可以通过定义一个类型为 `ChatHistoryAgentThread` 的线程来运行代理。任何必要的系统消息都可以通过 `invoke_stream` 的 `messages` 关键字参数提供给代理。

定义完这些后，我们创建一个 `user_inputs`，它代表用户发送给代理的内容。在这个例子中，我们将消息设置为 `Plan me a sunny vacation`。

您可以随意更改此消息，看看代理会有怎样不同的回应。


In [11]:
# -----------------------------------------------------
# 4. 异步主程序和执行逻辑
# -----------------------------------------------------
async def main():
    # Create a new thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    # 初始化对话线程（Thread），用于保存每一次聊天的历史记录
    thread: ChatHistoryAgentThread | None = None

    # 定义用户输入列表
    user_inputs = [
        "Plan me a day trip.",
        "I don't like that destination. Plan me another vacation.",
    ]

    # 遍历所有用户输入
    for user_input in user_inputs:
        print(f"# User: {user_input}\n")
        first_chunk = True  # 标记是否为流式响应的第一个块（用于格式化输出）

        # 代理的核心调用：使用 invoke_stream 进行流式调用，以实现实时输出
        async for response in agent.invoke_stream(
            messages=user_input, # 当前的用户消息
            thread=thread, # 传入当前的线程，确保对话连续性
        ):
            # 5. Print the response
            # 打印流式响应的逻辑
            if first_chunk:
                print(f"# {response.name}: ", end="", flush=True)
                first_chunk = False

            # 打印当前流块的内容
            print(f"{response}", end="", flush=True)
            # 关键：更新线程对象，确保下一轮循环使用最新的对话状态
            thread = response.thread
        print()

    # Clean up the thread
    # 清理线程 (如果线程存在)
    # 这一步通常用于删除云端存储的对话历史，以节省资源
    await thread.delete() if thread else None

# 运行异步主函数
await main()

# User: Plan me a day trip.

# TravelAgent: For your day trip, how about visiting Berlin, Germany? It's a city rich with history and culture. We can plan to visit key attractions such as the Brandenburg Gate, the remnants of the Berlin Wall, and the historic Reichstag building. There are also many museums and a vibrant food scene to explore. Would you like to proceed with planning activities in Berlin or would you prefer a different location?
# User: I don't like that destination. Plan me another vacation.

# TravelAgent: Let's plan a different vacation for you. How about a trip to Paris, France? The city is renowned for its romantic ambiance, art, and architecture. For your day trip, we could include a visit to the iconic Eiffel Tower, a stroll along the Seine, and perhaps a tour of the Louvre Museum if you're interested in art. There

CancelledError: 


---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
